# Baseline Modeling and Exploratory Data Analysis

This notebook adds a transparent baseline and a reproducible EDA layer to the LIGO neural-posterior-estimation project. It does not retrain the expensive SNPE model. Instead, it loads the model-ready checkpoint created by `notebooks/main.ipynb`, audits it, builds interpretable waveform summaries, creates a leakage-conscious development split, and compares a closed-form ridge baseline across O3a and O4a.

**Notebook ownership:** one contributor owns and maintains this notebook. Code cells are explicitly numbered from 1 and should be executed in order before milestone submission.

**Expected artifact:** `artifacts/data/training_data/training_data.pkl`. This file is generated by Section 5 of `main.ipynb` and is intentionally not committed because it is large.


## 1. Environment and deterministic setup

In [ ]:
# Cell 1
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT = PROJECT_ROOT / "artifacts" / "data" / "training_data" / "training_data.pkl"
PARAM_NAMES = ["chirp_mass", "mass_ratio", "chi_eff", "distance"]
SAMPLE_RATE_HZ = 4096
SEED = 7

plt.style.use("seaborn-v0_8-whitegrid")
print("project root:", PROJECT_ROOT)
print("training artifact:", ARTIFACT)


## 2. Load the model-ready checkpoint

The checkpoint is trusted project output produced by `main.ipynb`. Python pickle files should never be loaded from an untrusted source.

In [ ]:
# Cell 2
if not ARTIFACT.exists():
    raise FileNotFoundError(
        f"Missing {ARTIFACT}. Run notebooks/main.ipynb through Section 5 first, "
        "or copy the validated training_data.pkl checkpoint into that location."
    )

with ARTIFACT.open("rb") as stream:
    data = pickle.load(stream)

required = {
    "X_train", "theta_train", "X_test_o3a", "theta_test_o3a",
    "X_test_o4a", "theta_test_o4a",
}
missing = required - set(data)
if missing:
    raise KeyError(f"Training checkpoint is missing keys: {sorted(missing)}")

arrays = {name: np.asarray(data[name]) for name in sorted(required)}
for name, value in arrays.items():
    print(f"{name:16s} shape={value.shape!s:18s} dtype={value.dtype}")


## 3. Data-quality audit

No missing-value imputation is used. Nonfinite values, unexpected dimensions, or inconsistent row counts are treated as pipeline failures rather than silently repaired.

In [ ]:
# Cell 3
expected_rows = {"train": 750, "test_o3a": 50, "test_o4a": 800}
audit_rows = []

for split in expected_rows:
    X = arrays[f"X_{split}"]
    theta = arrays[f"theta_{split}"]
    if X.shape[0] != theta.shape[0]:
        raise ValueError(f"Row mismatch in {split}: X={X.shape}, theta={theta.shape}")
    if X.shape[0] != expected_rows[split] or X.shape[1] != 16384 or theta.shape[1] != 4:
        raise ValueError(f"Unexpected shape in {split}: X={X.shape}, theta={theta.shape}")
    audit_rows.append({
        "split": split,
        "examples": X.shape[0],
        "samples/example": X.shape[1],
        "nonfinite_X": int(X.size - np.isfinite(X).sum()),
        "nonfinite_theta": int(theta.size - np.isfinite(theta).sum()),
        "X_mean": float(X.mean()),
        "X_std": float(X.std()),
        "X_abs_max": float(np.abs(X).max()),
    })

audit = pd.DataFrame(audit_rows)
display(audit)
assert (audit[["nonfinite_X", "nonfinite_theta"]].to_numpy() == 0).all()


## 4. Create an explicit development validation set

The original 750-row O3a development array is split into 600 training and 150 validation examples. The existing 50-row O3a and 800-row O4a test sets remain untouched. The current checkpoint does not preserve each example's `noise_key`, so a fully group-aware split by noise segment is not yet possible; this limitation is reported rather than hidden.

In [ ]:
# Cell 4
rng = np.random.default_rng(SEED)
order = rng.permutation(arrays["X_train"].shape[0])
validation_idx = order[:150]
training_idx = order[150:]

assert len(set(training_idx) & set(validation_idx)) == 0
assert len(training_idx) == 600 and len(validation_idx) == 150

X_dev_train = arrays["X_train"][training_idx]
theta_dev_train = arrays["theta_train"][training_idx]
X_validation = arrays["X_train"][validation_idx]
theta_validation = arrays["theta_train"][validation_idx]

split_summary = pd.DataFrame({
    "split": ["O3a development train", "O3a validation", "O3a held-out test", "O4a cross-run test"],
    "examples": [len(training_idx), len(validation_idx), len(arrays["X_test_o3a"]), len(arrays["X_test_o4a"])],
    "used_for_tuning": [False, True, False, False],
})
display(split_summary)


## 5. Target distributions and correlations

These plots reveal accepted-injection coverage, skew, boundary effects, and correlations among the four physical targets. O3a and O4a differences can arise from the SNR acceptance gate even when both runs draw from the same prior.

In [ ]:
# Cell 5
target_sets = {
    "O3a development": arrays["theta_train"],
    "O3a test": arrays["theta_test_o3a"],
    "O4a test": arrays["theta_test_o4a"],
}

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for j, (ax, name) in enumerate(zip(axes.flat, PARAM_NAMES)):
    for label, theta in target_sets.items():
        ax.hist(theta[:, j], bins=24, density=True, histtype="step", linewidth=1.6, label=label)
    ax.set_title(f"Distribution of {name}")
    ax.set_xlabel(name)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
plt.show()

corr = pd.DataFrame(arrays["theta_train"], columns=PARAM_NAMES).corr()
fig, ax = plt.subplots(figsize=(6.2, 5.2))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(4), PARAM_NAMES, rotation=30, ha="right")
ax.set_yticks(range(4), PARAM_NAMES)
for row in range(4):
    for col in range(4):
        ax.text(col, row, f"{corr.iloc[row, col]:.2f}", ha="center", va="center")
ax.set_title("O3a target correlation matrix")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.show()


## 6. Engineer interpretable waveform summaries

The baseline deliberately avoids all 16,384 raw samples. It uses time-domain scale/shape summaries and power in physically meaningful frequency bands. This provides an interpretable lower bound and enables conventional feature-importance analysis.

In [ ]:
# Cell 6
FEATURE_NAMES = [
    "mean", "std", "rms", "peak_abs", "crest_factor", "zero_crossing_rate",
    "q05", "q50", "q95", "bandpower_35_70", "bandpower_70_140",
    "bandpower_140_350", "spectral_centroid",
]

def waveform_features(X, sample_rate=SAMPLE_RATE_HZ, batch_size=64):
    rows = []
    frequencies = np.fft.rfftfreq(X.shape[1], d=1.0 / sample_rate)
    bands = [(35, 70), (70, 140), (140, 350)]
    for start in range(0, len(X), batch_size):
        batch = np.asarray(X[start:start + batch_size], dtype=np.float64)
        rms = np.sqrt(np.mean(batch**2, axis=1))
        peak = np.max(np.abs(batch), axis=1)
        spectrum = np.abs(np.fft.rfft(batch, axis=1))**2
        total_power = spectrum.sum(axis=1) + 1e-12
        bandpowers = [spectrum[:, (frequencies >= lo) & (frequencies < hi)].sum(axis=1) / total_power
                      for lo, hi in bands]
        centroid = (spectrum * frequencies).sum(axis=1) / total_power
        rows.append(np.column_stack([
            batch.mean(axis=1), batch.std(axis=1), rms, peak,
            peak / (rms + 1e-12),
            np.mean(np.diff(np.signbit(batch), axis=1), axis=1),
            np.quantile(batch, .05, axis=1), np.quantile(batch, .50, axis=1),
            np.quantile(batch, .95, axis=1), *bandpowers, centroid,
        ]))
    result = np.vstack(rows)
    if not np.isfinite(result).all():
        raise ValueError("Engineered feature matrix contains nonfinite values")
    return result

F_dev_train = waveform_features(X_dev_train)
F_validation = waveform_features(X_validation)
F_test_o3a = waveform_features(arrays["X_test_o3a"])
F_test_o4a = waveform_features(arrays["X_test_o4a"])
print("feature shapes:", F_dev_train.shape, F_validation.shape, F_test_o3a.shape, F_test_o4a.shape)


## 7. Feature distributions and feature-target correlations

In [ ]:
# Cell 7
feature_frame = pd.DataFrame(F_dev_train, columns=FEATURE_NAMES)
target_frame = pd.DataFrame(theta_dev_train, columns=PARAM_NAMES)
joined = pd.concat([feature_frame, target_frame], axis=1)
feature_target_corr = joined.corr().loc[FEATURE_NAMES, PARAM_NAMES]

fig, ax = plt.subplots(figsize=(8.5, 6.2))
image = ax.imshow(feature_target_corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(PARAM_NAMES)), PARAM_NAMES, rotation=25, ha="right")
ax.set_yticks(range(len(FEATURE_NAMES)), FEATURE_NAMES)
ax.set_title("Engineered-feature correlations with physical targets")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.show()

display(feature_frame.describe().T[["mean", "std", "min", "50%", "max"]])


## 8. Standardization and closed-form ridge baseline

In [ ]:
# Cell 8
feature_mean = F_dev_train.mean(axis=0)
feature_std = F_dev_train.std(axis=0)
feature_std[feature_std == 0] = 1
target_mean = theta_dev_train.mean(axis=0)
target_std = theta_dev_train.std(axis=0)
target_std[target_std == 0] = 1

def scale_features(F):
    return (F - feature_mean) / feature_std

def scale_targets(theta):
    return (theta - target_mean) / target_std

def ridge_fit(F, theta, alpha):
    design = np.column_stack([np.ones(len(F)), scale_features(F)])
    penalty = np.eye(design.shape[1])
    penalty[0, 0] = 0
    return np.linalg.solve(design.T @ design + alpha * penalty, design.T @ scale_targets(theta))

def ridge_predict(F, weights):
    design = np.column_stack([np.ones(len(F)), scale_features(F)])
    return (design @ weights) * target_std + target_mean


## 9. Tune regularization on validation data only

In [ ]:
# Cell 9
alphas = np.logspace(-4, 4, 33)
tuning_rows = []
for alpha in alphas:
    weights = ridge_fit(F_dev_train, theta_dev_train, alpha)
    prediction = ridge_predict(F_validation, weights)
    standardized_rmse = np.sqrt(np.mean(((prediction - theta_validation) / target_std) ** 2))
    tuning_rows.append({"alpha": alpha, "standardized_validation_RMSE": standardized_rmse})

tuning = pd.DataFrame(tuning_rows)
best_alpha = float(tuning.loc[tuning["standardized_validation_RMSE"].idxmin(), "alpha"])
print("best alpha:", best_alpha)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(tuning["alpha"], tuning["standardized_validation_RMSE"], marker="o", markersize=3)
ax.axvline(best_alpha, color="tab:red", linestyle="--", label=f"selected alpha={best_alpha:g}")
ax.set_title("Ridge hyperparameter selection on O3a validation data")
ax.set_xlabel("Ridge regularization alpha")
ax.set_ylabel("Mean standardized RMSE")
ax.legend()
plt.show()


## 10. Compare validation, within-run test, and cross-run test metrics

In [ ]:
# Cell 10
best_weights = ridge_fit(F_dev_train, theta_dev_train, best_alpha)

def regression_metrics(truth, prediction, split):
    rows = []
    for j, name in enumerate(PARAM_NAMES):
        error = prediction[:, j] - truth[:, j]
        denominator = np.sum((truth[:, j] - truth[:, j].mean()) ** 2)
        rows.append({
            "split": split,
            "target": name,
            "MAE": np.mean(np.abs(error)),
            "RMSE": np.sqrt(np.mean(error**2)),
            "R2": 1 - np.sum(error**2) / denominator,
        })
    return rows

evaluation_sets = {
    "O3a validation": (F_validation, theta_validation),
    "O3a held-out test": (F_test_o3a, arrays["theta_test_o3a"]),
    "O4a cross-run test": (F_test_o4a, arrays["theta_test_o4a"]),
}
metric_rows = []
predictions = {}
for split, (features, truth) in evaluation_sets.items():
    predictions[split] = ridge_predict(features, best_weights)
    metric_rows.extend(regression_metrics(truth, predictions[split], split))

metrics = pd.DataFrame(metric_rows)
display(metrics.round(4))
display(metrics.pivot(index="target", columns="split", values="R2").round(3))


## 11. Residual and cross-run error analysis

In [ ]:
# Cell 11
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
for j, (ax, target) in enumerate(zip(axes.flat, PARAM_NAMES)):
    for split, (_, truth) in evaluation_sets.items():
        residual = predictions[split][:, j] - truth[:, j]
        ax.hist(residual, bins=24, density=True, histtype="step", linewidth=1.5, label=split)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Residual distribution: {target}")
    ax.set_xlabel("Prediction - truth")
    ax.set_ylabel("Density")
    ax.legend(fontsize=7)
plt.show()


## 12. Permutation feature importance

Importance is the increase in standardized validation MSE when one feature is shuffled. Larger values indicate that the fitted baseline depends more strongly on that feature. This is model-specific importance, not a causal claim.

In [ ]:
# Cell 12
baseline_prediction = ridge_predict(F_validation, best_weights)
baseline_mse = np.mean(((baseline_prediction - theta_validation) / target_std) ** 2)
importance_rows = []

for feature_index, feature_name in enumerate(FEATURE_NAMES):
    increases = []
    for repeat in range(20):
        shuffled = F_validation.copy()
        permutation = np.random.default_rng(SEED + repeat).permutation(len(shuffled))
        shuffled[:, feature_index] = shuffled[permutation, feature_index]
        prediction = ridge_predict(shuffled, best_weights)
        mse = np.mean(((prediction - theta_validation) / target_std) ** 2)
        increases.append(mse - baseline_mse)
    importance_rows.append({
        "feature": feature_name,
        "mean_MSE_increase": np.mean(increases),
        "std_MSE_increase": np.std(increases),
    })

importance = pd.DataFrame(importance_rows).sort_values("mean_MSE_increase")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(importance["feature"], importance["mean_MSE_increase"],
        xerr=importance["std_MSE_increase"], color="tab:blue", alpha=.85)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Permutation importance on the O3a validation set")
ax.set_xlabel("Increase in mean standardized MSE")
ax.set_ylabel("Engineered feature")
plt.show()

display(importance.sort_values("mean_MSE_increase", ascending=False).head(8))


## Interpretation checklist

After executing all cells, update this section with observed values rather than generic claims:

1. Which targets are skewed or truncated after the SNR acceptance gate?
2. Which engineered summaries correlate most strongly with each target?
3. Does ridge regression outperform a mean-prediction baseline (R2 > 0)?
4. How much does each target's error change from O3a validation/test to O4a?
5. Do residuals show bias, heavy tails, or boundary effects?
6. Which features have stable positive permutation importance?

### Known limitation and recommended pipeline change

The processed checkpoint omits source `noise_key` identifiers. A future checkpoint version should preserve a group ID for every example so validation can hold out entire noise segments and rule out background leakage. Until then, the fixed O3a validation split in this notebook is deterministic and disjoint by row, but not guaranteed disjoint by originating noise segment.
